> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 岗位学习课：输出隔离到outputs/public_learning/，版本带public-learning，动态选取示例客户。不能替换已发给团队的H02_candidate_v2。按02→03→04运行。

# 04｜从“程序跑完”到“可以交接”：H02 独立复核与教学

本课接续 02（明细→订单→客户）和 03（时间覆盖→特征→标签）。**不重新定义任务，不把候选方案冒充已确认方案。** 你已授权使用三个已见探索日期。本课从原始 orders、customers、items 重新计算，检查交接文件是否能被独立重建，并输出 v2 候选包。

你要能够解释：① 为什么行数变了；② 哪些人被排除、为何排除；③ 特征与标签分别用了什么时间；④ 0 标签何时才有意义；⑤ 哪些检查通过，哪些判断仍需团队确认。

学习路线：原始文件与版本 → 粒度与连接 → 资格 → 时间 → 全量复算 → 个案追溯 → 可用时点 → 验收与交接。每次先读问题，再运行代码、看输出，最后用自己的话回答。

本课输出到新的 `outputs/public_learning/H02_candidate_v2`；v1 保留。v2 数值应与 v1 一致，变化是数据版本和更完整的验收证据。最终评价日期仍未选定，最终标签未生成。

## 1｜首先知道我们到底在检查哪一份文件

哈希用于判断字节是否一致，不能证明业务定义正确。既要核输入指纹，也要核计算规则。下面检查 v1 的全部已登记产物，以及原始来源清单。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json, hashlib, shutil, sys
from zipfile import ZipFile, ZIP_DEFLATED
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/source-manifest.json').exists())
BASE=ROOT/'outputs/public_learning/H02_candidate_v1'
OUT=ROOT/'outputs/public_learning/H02_candidate_v2'
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'audit').mkdir(exist_ok=True)
VERSION='H02-public-learning-v2'
KEY=['customer_unique_id','snapshot_date']
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
    return h.hexdigest()
def save(frame,name):
    target=OUT/name
    target.parent.mkdir(parents=True,exist_ok=True)
    frame.to_csv(target,index=False,encoding='utf-8-sig')
def table(frame):return frame.to_csv(index=False)
previous=json.loads((BASE/'data-manifest.json').read_text(encoding='utf-8'))
for a in previous['artifacts']: assert sha(BASE/a['file'])==a['sha256'],a['file']
sources=json.loads((ROOT/'data/source-manifest.json').read_text(encoding='utf-8'))
assert sha(ROOT/'data/olist-v2.zip')==sources['zip_sha256']
for a in previous['input_hashes']:assert sha(ROOT/'data/raw'/a['file'])==a['sha256'],a['file']
features=pd.read_csv(BASE/'snapshot_features.csv',dtype={'customer_unique_id':'string'})
labels=pd.read_csv(BASE/'snapshot_labels.csv',dtype={'customer_unique_id':'string'})
cohorts=pd.read_csv(BASE/'coverage-and-cohorts.csv')
feature_names=previous['feature_columns']
assert set(labels.snapshot_date)=={'2017-10-01','2018-01-01','2018-04-01'}
print('已核 v1 文件数：',len(previous['artifacts']))
print('特征行数：',len(features),'；标签行数：',len(labels))
display(cohorts[['snapshot_date','split','history_start','history_end_exclusive','label_start','label_end_exclusive']])

## 2｜为什么再读原始数据？

仅拿保存的特征表检查“没有重复”不够：即使金额重复算了，键仍可能唯一。这里独立读取原始三张表，先确认实际粒度，再重建订单金额，避免只重复检查同一份错误中间表。

orders 一行一订单；customers 一行一个订单侧客户编号；items 一行一个订单内商品明细编号。买家唯一编号可以重复，因为一人能有多单。

In [ ]:
raw=ROOT/'data/raw'
o=pd.read_csv(raw/'olist_orders_dataset.csv',dtype={'order_id':'string','customer_id':'string'})
c=pd.read_csv(raw/'olist_customers_dataset.csv',dtype={'customer_id':'string','customer_unique_id':'string'})
i=pd.read_csv(raw/'olist_order_items_dataset.csv',dtype={'order_id':'string','order_item_id':'int64'})
for field in ['order_approved_at','order_purchase_timestamp','order_delivered_customer_date']:
    o[field]=pd.to_datetime(o[field],errors='raise')
checks=[]
for name,frame,keys in [('orders',o,['order_id']),('customers',c,['customer_id']),('items',i,['order_id','order_item_id'])]:
    nulls=int(frame[keys].isna().any(axis=1).sum())
    duplicates=int(frame.duplicated(keys).sum())
    checks.append({'table':name,'rows':len(frame),'key':'+'.join(keys),'null_key_rows':nulls,'duplicate_extra_rows':duplicates})
    assert nulls==duplicates==0
assert c.customer_unique_id.notna().all()
assert i[['price','freight_value']].notna().all().all()
assert i[['price','freight_value']].ge(0).all().all()
display(pd.DataFrame(checks))
print('订单侧客户编号：',c.customer_id.nunique(),'；匿名买家：',c.customer_unique_id.nunique())

## 3｜先汇总再连接，并解释每一次行数变化

`groupby` 把同一订单明细放在一起；金额求和，明细数计行。多行变一行是汇总，不是删掉了数据。之后以 orders 为左表：没有商品匹配的订单也保留，金额为缺失，不能把它当作免费订单填 0。

客户连接用 customer_id；跨单聚合买家才用 customer_unique_id。下面的断言还检查商品表是否有孤立订单号，避免金额因左连接悄悄消失。

In [ ]:
item_order=i.groupby('order_id',as_index=False).agg(
    amount=('price','sum'),freight=('freight_value','sum'),item_rows=('order_item_id','size'))
assert item_order.order_id.is_unique
assert i.order_id.isin(o.order_id).all()
joined=o.merge(item_order,on='order_id',how='left',validate='one_to_one',indicator='item_link')
joined=joined.merge(c[['customer_id','customer_unique_id','customer_state']],on='customer_id',how='left',validate='one_to_one',indicator='customer_link')
assert len(joined)==len(o) and joined.order_id.is_unique
assert joined.customer_link.eq('both').all()
assert np.isclose(item_order.amount.sum(),i.price.sum(),rtol=1e-12)
assert np.isclose(joined.amount.sum(),i.price.sum(),rtol=1e-12)
assert np.isclose(joined.freight.sum(),i.freight_value.sum(),rtol=1e-12)
approved=joined.loc[joined.order_approved_at.notna()].copy()
eligible=approved.loc[approved.amount.gt(0)].copy()
stages=pd.DataFrame([
    ['原始订单',len(o),0,0,0],
    ['商品明细汇总到订单',len(item_order),0,0,len(i)-len(item_order)],
    ['订单左连商品汇总',len(joined),int(joined.item_link.ne('both').sum()),0,0],
    ['再连客户',len(joined),int(joined.customer_link.ne('both').sum()),0,0],
    ['要求批准时间',len(approved),0,len(joined)-len(approved),0],
    ['再要求商品金额大于0',len(eligible),0,len(approved)-len(eligible),0],
],columns=['stage','rows','unmatched_rows','excluded_orders','collapsed_item_rows'])
display(stages)
print('未匹配商品金额仍缺失：',joined.loc[joined.item_link.ne('both'),'amount'].isna().all())

**如何读这张表：**775 笔订单没有商品明细匹配，但连接时仍保留；资格筛选先排除无批准时间的 160 笔，再从剩余订单排除金额不满足的 629 笔，得到 98,652 笔合格事件。775 与 160、629 不是可以直接相加的互斥类别，筛选顺序决定分母。

金额规则是已授权的课程研究定义；不是说这些被排除的原始记录“都是错误”。我们保留原始数据与原因，不删除源文件。没有按最终 delivered 状态筛资格。

## 4｜先画清楚时间，才能谈标签

每个 t：历史为 `[t−6个自然月,t)`；标签为 `[t,t+90天)`。两个窗口长度不必相同。标签起点订单计入未来，标签终点订单不计入本次未来。训练标签必须在下个评分时点可获得；这里采用事件及时入库的假设，实际入库延迟待核。

三期都是已见探索数据。开发训练/验证是工作分工，不能把已经看过的 4 月改名为独立最终测试。90 天是暂定任务期限，不是本课证明出的最优值。

In [ ]:
time_checks=[]
for row in cohorts.itertuples():
    t=pd.Timestamp(row.snapshot_date); left=t-pd.DateOffset(months=6); right=t+pd.Timedelta(days=90)
    assert left==pd.Timestamp(row.history_start)
    assert right==pd.Timestamp(row.label_end_exclusive)
    assert left>=pd.Timestamp(previous['observation_start']) and right<=pd.Timestamp(previous['observation_end_exclusive'])
    time_checks.append({'snapshot_date':row.snapshot_date,'history_start':str(left.date()),'history_end_exclusive':str(t.date()),
                        'label_start':str(t.date()),'label_end_exclusive':str(right.date()),'label_mature_at_assumed':str(right.date()),'final_test':False})
times=pd.DataFrame(time_checks)
for a,b in zip(times.to_dict('records'),times.to_dict('records')[1:]):
    assert pd.Timestamp(a['label_mature_at_assumed'])<=pd.Timestamp(b['snapshot_date'])
display(times)

## 5｜从原始三张表独立复算全部特征与标签

此处不调用 03 的 make_snapshot，不读取它的订单事实表。计算时先以历史合格订单固定名单；未来新买家不会加进来。

R 是距最近批准的天数；F 是不同订单数；M 是商品金额。平均客单价 M/F；运费占比为运费/(商品金额+运费)。最近30天没有订单，活动数可以为0。观察历史长度不是客户注册年龄。州取历史最后一笔非空州，缺失用 UNKNOWN。

比较每一列：ID、计数、类别精确相等；浮点数允许计算机舍入误差。全部一致只能证明两条计算路径在此口径下可复现，不能证明业务假设正确。

In [ ]:
result=[]
recomputed=[]
for date in cohorts.snapshot_date:
    t=pd.Timestamp(date)
    h=eligible.loc[(eligible.order_approved_at>=t-pd.DateOffset(months=6))&(eligible.order_approved_at<t)].copy()
    f=eligible.loc[(eligible.order_approved_at>=t)&(eligible.order_approved_at<t+pd.Timedelta(days=90))]
    g=h.groupby('customer_unique_id')
    a=pd.DataFrame(index=g.size().index)
    a['F']=g.order_id.nunique()
    a['M']=g.amount.sum()
    a['R']=(t-g.order_approved_at.max()).dt.total_seconds()/86400
    a['average_order_value']=a.M/a.F
    a['max_order_value']=g.amount.max()
    a['average_item_rows']=g.item_rows.mean()
    a['freight_share']=g.freight.sum()/(a.M+g.freight.sum())
    a['observed_history_days']=(t-g.order_approved_at.min()).dt.total_seconds()/86400
    recent=h.loc[h.order_approved_at>=t-pd.Timedelta(days=30)].groupby('customer_unique_id')
    a['recent30_orders']=recent.order_id.nunique().reindex(a.index,fill_value=0)
    a['recent30_spend']=recent.amount.sum().reindex(a.index,fill_value=0)
    latest=h.dropna(subset=['customer_state']).sort_values(['order_approved_at','order_id']).drop_duplicates('customer_unique_id',keep='last')
    a['state']=latest.set_index('customer_unique_id').customer_state.reindex(a.index).fillna('UNKNOWN')
    a['y']=a.index.isin(f.customer_unique_id).astype(int)
    a=a.reset_index();a['snapshot_date']=date
    expected=features.loc[features.snapshot_date.eq(date)].merge(labels.loc[labels.snapshot_date.eq(date),KEY+['y']],on=KEY,validate='one_to_one')
    pair=a.merge(expected,on=KEY,how='outer',validate='one_to_one',suffixes=('_raw','_file'),indicator=True)
    assert pair._merge.eq('both').all()
    for field in feature_names+['y']:
        if field in ['state','F','recent30_orders','y']:
            assert (pair[field+'_raw']==pair[field+'_file']).all(),(date,field)
            max_error=0.0
        else:
            np.testing.assert_allclose(pair[field+'_raw'],pair[field+'_file'],rtol=1e-10,atol=1e-8,err_msg=f'{date}:{field}')
            max_error=float((pair[field+'_raw']-pair[field+'_file']).abs().max())
        result.append({'snapshot_date':date,'field':field,'checked_rows':len(pair),'max_absolute_error':max_error,'status':'passed'})
    recomputed.append(a)
comparison=pd.DataFrame(result)
display(comparison)
assert sum(len(a) for a in recomputed)==len(features)
print('全部客户快照、11个特征及标签复算一致。')

## 6｜你亲手看懂一个多单客户，而不只看 PASS

选择 2018-01-01 快照的真实多单客户。先打印其历史订单和原始商品明细，再计算 F/M/运费。你应先用纸笔求和，再看程序答案。明细多行不是多笔订单，F 必须数不同 order_id。

In [ ]:
buyer=features.loc[features.snapshot_date.eq('2018-01-01') & features.F.gt(1),'customer_unique_id'].sort_values().iloc[0]
t=pd.Timestamp('2018-01-01')
history=eligible.loc[eligible.customer_unique_id.eq(buyer)&eligible.order_approved_at.ge(t-pd.DateOffset(months=6))&eligible.order_approved_at.lt(t)]
detail=i.loc[i.order_id.isin(history.order_id),['order_id','order_item_id','price','freight_value']]
record=features.loc[features.customer_unique_id.eq(buyer)&features.snapshot_date.eq('2018-01-01')]
assert len(record)==1
assert history.order_id.nunique()==record.iloc[0].F
assert np.isclose(detail.price.sum(),record.iloc[0].M)
display(history[['order_id','order_approved_at','amount','freight','item_rows']])
display(detail)
display(record[KEY+['R','F','M','average_order_value','freight_share']])
print('手算核对：F=',history.order_id.nunique(),'，M=',detail.price.sum())
save(history[['order_id','customer_unique_id','order_approved_at','amount','freight','item_rows']],'audit/independent-multibuyer-orders.csv')
save(detail,'audit/independent-multibuyer-items.csv')

**读结果：**公开版在运行时选一名多单客户，实际笔数和金额以本格输出为准。金额不含运费。把一行客户特征追溯回历史订单，再追溯到原始商品明细。

## 7｜边界订单：为什么恰好在快照时点算未来？

用真实订单的批准时刻作为临时分界，只核成员归属，不输出这个临时日期的新客户标签。快照前一秒、恰好快照、后一秒的归属不能混淆。03 中另有六个人工边界、未来字段变动与覆盖不足拒绝测试。

In [ ]:
boundary_id=eligible.loc[eligible.order_approved_at.ge('2018-01-01') & eligible.order_approved_at.lt('2018-02-01'),'order_id'].sort_values().iloc[0]
event=eligible.loc[eligible.order_id.eq(boundary_id)].iloc[0]
bt=event.order_approved_at
membership=[]
for cut in [bt,bt+pd.Timedelta(seconds=1)]:
    is_hist=bool(cut-pd.DateOffset(months=6)<=bt<cut)
    is_future=bool(cut<=bt<cut+pd.Timedelta(days=90))
    membership.append({'order_id':boundary_id,'approved_at':str(bt),'test_cut':str(cut),'in_history':is_hist,'in_future':is_future,'amount':event.amount})
boundary=pd.DataFrame(membership)
assert boundary.in_history.tolist()==[False,True]
assert boundary.in_future.tolist()==[True,False]
display(boundary)

## 8｜覆盖检查：有记录不等于证明完整，零记录也不等于丢失

原始购买时间的最大值不能代替合格批准事件覆盖。下面重算全部月份，并展示 2018 年 4 月异常与 9 月尾部。工作结束边界暂为 2018-09-01（不含），并非证明此前每天源系统都完整。

4月21–23日有641笔下单但无批准；这些订单全部于其后获批。可观察到批准记录延后集中出现，不能从导出文件确定真实原因。保留原批准时间，不把它替换成购买时间，不删除这些天，也不补造订单。

In [ ]:
purchase=o.groupby(o.order_purchase_timestamp.dt.to_period('M')).size().rename('purchase_orders')
approval=o.dropna(subset=['order_approved_at']).groupby(o.order_approved_at.dt.to_period('M')).size().rename('approved_orders')
qualified=eligible.groupby(eligible.order_approved_at.dt.to_period('M')).size().rename('eligible_approved_orders')
months=pd.concat([purchase,approval,qualified],axis=1).fillna(0).astype(int)
months.index=months.index.astype(str);months.index.name='month'
display(months)
gap=o.loc[o.order_purchase_timestamp.ge('2018-04-21')&o.order_purchase_timestamp.lt('2018-04-24')]
gap_dates=gap.order_approved_at.dt.strftime('%Y-%m-%d').fillna('MISSING').value_counts().sort_index()
display(gap_dates.rename('orders').to_frame())
display(cohorts[['snapshot_date','history_zero_days','label_zero_days','coverage_warning']])
print('所有三期均为条件性探索范围；源系统完整性和实际入库延迟仍待核。')

## 9｜哪些字段可以作为当时的输入？

已送达状态、最终评分、实际送达时间不能整体搬入提前预测。03 已统计出大量历史订单在快照后才送达或填写评价；这些字段没有进入初版特征。

批准时间有事件时刻，但商品金额、运费、客户州是静态导出值，没有修改历史。我们依赖“批准时已有这些交易信息，后续未发生影响本分析的修改”的假设。不能把这个假设说成已用系统日志证明。真正上线前需源系统核验。

字段清单明确包含11个输入；客户ID、快照日、split、版本只是连接元数据，不作为模型输入；y只在标签文件。此处核查实际列，而非口头约定。

In [ ]:
assert set(features.columns)==set(KEY+['split','data_version']+feature_names)
assert 'y' not in features.columns
for forbidden in ['order_status','review_score','order_delivered_customer_date']:
    assert forbidden not in feature_names
assert not features.duplicated(KEY).any() and not labels.duplicated(KEY).any()
aligned=features[KEY].merge(labels[KEY],on=KEY,how='outer',indicator=True,validate='one_to_one')
assert aligned._merge.eq('both').all()
display(pd.read_csv(BASE/'audit/field-time-audit.csv'))
print('键一致；11项输入不包含最终状态、送达结果、评分或标签。')

## 10｜把“检查发现→处理→影响→证据”写成处理记录

不同问题不能用同一套删行或填零。主键重复应先定位；连接缺口保留并追溯；无批准时间按当前资格排除；窗口覆盖不足禁止生成确定的0。下面是你的可复查处理记录，而不是泛泛的“已清洗”。

In [ ]:
processing=pd.DataFrame([
    ['键和粒度','订单/客户键及明细复合键无空无重','未删除记录','订单99441，明细112650','audit/independent-key-checks.csv','passed'],
    ['一对多金额','订单可有多条商品明细','先按order_id汇总金额和运费，再左连','连接后订单行数99441；总额守恒','audit/independent-join-stages.csv','passed'],
    ['商品匹配缺口','775笔订单没有明细匹配','连接保留，金额NA；按资格另筛','不将缺失商品金额填0','audit/step1-excluded-orders.csv','passed'],
    ['资格筛选','160笔无批准时间，之后629笔金额不满足','顺序排除，保留源记录和原因','剩98652笔合格订单','audit/independent-join-stages.csv','passed'],
    ['时间边界','历史和标签有不同的开闭边界','[t-6月,t)，[t,t+90天)','未因边界重复计单','audit/independent-real-boundary.csv','passed'],
    ['完整性异常','4月21至23日无批准、但有下单','保持批准时间；追踪随后获批','4月只作探索，原因待核','audit/april-gap-purchase-cohort-approval-dates.csv','pending_source_confirmation'],
    ['尾部稀疏','9月批准记录骤减、10月无批准','暂用9月1日前工作范围；不造0标签','未建立覆盖不足的新快照','audit/monthly-coverage.csv','conditional'],
    ['时点可用性','快照后送达及填写评价记录存在','不输入最终状态、送达结果、评分','11项输入仍依赖静态交易字段假设','audit/field-time-audit.csv','conditional'],
    ['最终评价','暂无核定的最终日期与标签开放要求','未生成新最终标签','三期均已探索，不能冒充未触碰测试','FINAL_LABEL_POLICY.md','pending_H01'],
],columns=['topic','finding','action','impact','evidence','status'])
display(processing)

## 11｜验收不是把所有项目勾成“完成”

本地计算核查可以完成；团队确认、接收方回执和源系统证明不能由代码代替。我们先交候选、写清已核范围和待核原因，接收方可选择暂收部分可用。下一次状态回报目标沿用岗位给出的2026-11-08 21:00（香港/上海），不是虚构的新约定；正式修订交付时刻待H01到位后与团队确定。

收到回执后修改相应记录；这里不会伪造已发送、已接收或已验收。

In [ ]:
acceptance=pd.DataFrame([
    ['操作1','键/粒度/金额/连接/资格/客户聚合','passed','02及本课2–6节'],
    ['操作2-计算','特征/标签/各期日期/每日每月/人数正例率','passed_for_candidate','03及本课4–5、8节'],
    ['操作2-完整性','源系统完整性、4月批准异常、尾部边界','conditional_pending','工作范围内探索，不宣称完整性已证明'],
    ['操作3-核查','真实边界、多单、随机、未来字段检查','passed_with_availability_assumptions','03的15个客户案例及本课全量复算'],
    ['操作3-最终标签','单独保管至冻结','not_applicable_yet','最终日期未定，标签未生成'],
    ['交付1','五项核心文件、生成命令、哈希、交接单','prepared_not_sent','本地v2候选包'],
    ['交付2-键核查','唯一键、特征标签一对一、可复算','passed','全部88341条快照复核'],
    ['交付2-回执','2–6号实际回执及修改闭环','pending_receipts','receipt-log.csv待实际回复'],
    ['交付3','已核范围、原因、下一状态时间、不造0','documented','H01与完整性仍待核；11月8日21点内部回报目标'],
],columns=['requirement','scope','status','evidence_or_gap'])
display(acceptance)

## 12｜导出 v2：数据数值不变，版本与核查证据更新

v2 保留 v1 所有核查附件，增加独立复算、处理记录和验收表。所有候选快照继续明确标为探索。接收方先选定一个版本，不能将 v1 特征和 v2 标签混接，即便本次值一致。

交接表给出每期筛选前后人数。筛选前的正例数沿用同一未来事件定义，仅放宽历史金额资格；不是同时更改标签定义。

In [ ]:
for path in (BASE/'audit').iterdir():
    if path.is_file():shutil.copy2(path,OUT/'audit'/path.name)
features['data_version']=VERSION;labels['data_version']=VERSION
save(features,'snapshot_features.csv');save(labels,'snapshot_labels.csv');save(cohorts,'coverage-and-cohorts.csv')
save(pd.DataFrame(checks),'audit/independent-key-checks.csv')
save(stages,'audit/independent-join-stages.csv');save(comparison,'audit/independent-all-features-labels.csv')
save(boundary,'audit/independent-real-boundary.csv');save(times,'audit/independent-time-design.csv')
save(processing,'processing-log.csv');save(acceptance,'acceptance-checklist.csv')
for name in ['data-dictionary.md','feature-columns.json','FINAL_LABEL_POLICY.md']:
    text=(BASE/name).read_text(encoding='utf-8').replace(previous['data_version'],VERSION)
    (OUT/name).write_text(text,encoding='utf-8')
receipts=pd.DataFrame([{'handoff_id':'H02','recipient_role':role,'sent_at':'','received_at':'','status':'未发送，待本人核验',
                        'input_version':VERSION,'checked_keys':'','issue':'','owner':'','next_action':'本人核验后通过组内私有空间交接'} for role in [2,3,4,5,6]])
save(receipts,'receipt-log.csv')
impact=pd.DataFrame([
    ['客户资格/批准口径/历史或未来窗','H03,H04,H05,H06,H07','重建H02；EDA、训练评估、分群、反馈与汇报数字核重跑'],
    ['特征公式/缺失处理','H03,H04,H05,H06,H07','先确认受影响的特征及模型/聚类；按新版本重算'],
    ['仅开放标签修正','H03,H04,H06,H07；H05有条件','重算标签相关统计、选模与评价；无监督输入没变时不必自动重聚类'],
    ['仅文档错字，数据与规则不变','文档使用方','更新文档版本和哈希；通常无需重训'],
],columns=['trigger','affected_handoffs','required_review'])
save(impact,'downstream-rerun-impact.csv')
print('已导出新版本：',VERSION)

## 13｜准备一张可给3、6号核对的定义表

“核口径”不是只问90天对不对。要一起确认：决策者、行动容量、预测对象、历史资格、事件时刻、标签、观察边界、最终评价安排，以及输入字段什么时候可见。下面整理的是待核定义表，不声称组员已经确认。

In [ ]:
definition="""# H02 数据定义与H01待核表

状态：用户授权的候选沿用版；3、6号正式回核待收到。

| 项目 | 当前工作定义 | 核对重点 |
|---|---|---|
| 决策者与用途 | CRM/留存运营经理，评审有限名额的已有客户 | 预测自然复购可能性，不等于触达增量 |
| 名单容量 | 每期K=ceil(0.10*N)，同分按匿名买家ID升序 | 由3、6号确认，H02不计算模型名单 |
| 统计单位 | customer_unique_id + snapshot_date，一人一期一行 | 3、4号同资格、同版本 |
| 合格事件 | order_approved_at非空且订单商品price合计>0 | 不依最终送达状态筛选 |
| 历史资格 | [t-6个自然月,t)至少一笔合格事件 | 6个月不是固定180天 |
| 事件时刻 | 原始order_approved_at，无时区字段 | 不擅自转香港时区；交件时间才用香港/上海 |
| 标签 | [t,t+90天)至少一笔同口径事件为1 | 覆盖满足工作假设、无事件才可记0 |
| 特征 | 11项历史交易字段 | 静态金额/州历史不变、及时可用为待核假设 |
| 日期 | 2017-10-01开发训练，2018-01-01开发验证，2018-04-01已见探索 | 均非未触碰最终测试 |
| 覆盖 | 暂用[2017-01-01,2018-09-01) | 4月异常与源系统完整性待核 |
| 标签成熟 | t+90天，假设无额外入库延迟 | 若延迟存在，应增加等待间隔 |
| 最终评价 | 尚未核定；未生成最终标签 | 3、6号给日期、冻结与开放条件 |
| 下游预处理 | 仅开发训练拟合，验证只转换 | 不把各期合并拟合后再声称时间验证 |

下一状态回报目标：2026-11-08 21:00（香港/上海，沿用岗位内部目标）。这是状态回报点，不代表待核项会在此时自动解决。正式修订交付时刻待H01输入到位后约定。
"""
(OUT/'data-definition-review.md').write_text(definition,encoding='utf-8')
display(Markdown(definition))

## 14｜把结果写成真正可核查的交接单

交接单列清：范围、计数、检查、缺口、生成命令、需收方核对什么。核心文件哈希随包提供，manifest自己的哈希放在独立CHECKSUMS文件中，避免让文件包含自身指纹的循环。

下面只生成本地文档，未发送消息。接收方的状态由实际回执更新。

In [ ]:
command=r'.\.venv312\Scripts\python.exe scripts/build_h02_reviewed.py'
count_lines=['| 快照 | 筛选前客户 | 筛选后客户 | 正例 | 复购率 | 历史起点 | 标签结束(不含) |',
             '|---|---:|---:|---:|---:|---|---|']
for r in cohorts.itertuples():
    count_lines.append(f'|{r.snapshot_date}|{r.customers_before_amount_rule}|{r.customers}|{r.positives}|{r.repurchase_rate:.4%}|{r.history_start}|{r.label_end_exclusive}|')
summary='\n'.join(count_lines)
handoff=f"""# H02 候选交接单｜{VERSION}

1号 → 2–6号。状态：本地准备完成，待成员本人核验；尚未发送、无接收方回执。

## 范围与人数

仅三个已见探索日期，无新的最终评价标签。共{len(features):,}条客户快照、{features.customer_unique_id.nunique():,}位匿名买家。重复跨期客户不是独立样本。所有日期使用源记录口径。

{summary}

从99441笔订单中，先排除160笔无批准时间，再排除629笔商品金额不满足，剩98652笔合格事件；各历史窗口再筛选与客户聚合。完整过程见processing-log.csv、coverage-and-cohorts.csv及audit。

## 计算与验收

订单、客户与明细键通过；商品额与运费先汇总、连接不增行、金额守恒。所有88341条快照的11项特征及标签由原始三表独立复算一致，特征/标签键一对一。真实边界、多单客户、15个抽查客户与人工边界证据在audit。

文件包括五项核心交付、字段清单、处理记录、验收表、口径待核表、回执表、影响表、复跑源文件及生成命令。各文件哈希见data-manifest.json；五项核心文件（含manifest）的哈希见CHECKSUMS.sha256。

## 待核与不可宣称事项

H01正式时间方案/最终评价仍待3、6号确认。4月21–23日有641笔下单但无批准，这些订单随后全部获批；保留批准时间，异常原因和源系统完整性待核。尾部9月稀疏，不用零标签补齐覆盖不足快照。原始金额/客户州的时点可用性依赖静态记录假设。三期均只供有条件的探索复算，不能标为正式完整窗口或未触碰最终测试。

最终评价日期未选定，所以最终标签未生成，不能称已封存某份测试标签。选定后须在包外私有存放，冻结前不共享逐行标签或未开放正例数。

## 接收方动作与下一状态时间

2号核缺失、分布、分母；3号核日期、标签成熟与输入列；4号核同一客户快照及训练拟合范围；5号核组合键与版本；6号协调H01和最终评价。请实际返回“已验收／退回补件／暂收部分可用”，附版本、已核范围与问题。回执填入receipt-log.csv。

下一状态回报目标2026-11-08 21:00（香港/上海，沿用内部H02期限），若H01未到则回报已核范围及缺口。正式修订版本的具体交付时间待收到H01后约定，不虚构已确认时间。改口径后按downstream-rerun-impact.csv核定H03–H07影响范围。

## 复跑

项目根目录执行：`{command}`。先使用仓库Python3.12环境、已注册olist312内核、已校验解压的原始数据。入口执行02、03、04并保存Notebook输出。原始CSV不修改。

11月13日Slides/英文讲稿/报告要点尚未在本包中完成，应以团队核定后的结果制作。
"""
(OUT/'HANDOFF.md').write_text(handoff,encoding='utf-8')
(OUT/'RUN.md').write_text('# 复跑与阅读\n\n在完整同源仓库根目录运行：\n\n```powershell\n'+command+'\n```\n\n使用Python3.12环境，安装仓库requirements.txt，注册olist312内核，先运行extract_data.py核验数据。入口执行02、03、04。包内reproduce含教学源文件，执行结果保存在仓库notebooks目录。\n\n若迁移到另一台电脑，将reproduce/notebooks置于仓库notebooks，将reproduce/scripts置于仓库scripts，保持原始data结构，使用目标机器实际的Python路径。\n\n学习顺序02→03→04。04对全量数据独立复算。CSV仅在组内私有空间交接。最终日期未定，无最终标签输出。\n',encoding='utf-8')
display(Markdown(summary))

## 15｜封装之前再核一次文件与版本

回读验证不是重新跑模型，而是确认保存的文件没有损坏、键和版本对得上。生成清单记录输入指纹、代码指纹、时间、测试结果和局限。清单不声称人工验收通过。

In [ ]:
saved_x=pd.read_csv(OUT/'snapshot_features.csv',dtype={'customer_unique_id':'string'})
saved_y=pd.read_csv(OUT/'snapshot_labels.csv',dtype={'customer_unique_id':'string'})
pd.testing.assert_frame_equal(saved_x,features,check_dtype=False,rtol=1e-10,atol=1e-8)
pd.testing.assert_frame_equal(saved_y,labels,check_dtype=False)
assert saved_x.data_version.eq(VERSION).all() and saved_y.data_version.eq(VERSION).all()
repro=OUT/'reproduce'
(repro/'notebooks').mkdir(parents=True,exist_ok=True);(repro/'scripts').mkdir(exist_ok=True)
names=['02-role01-order-fact-and-history.ipynb','03-role01-h02-coverage-features-labels.ipynb','04-role01-h02-independent-review.ipynb']
for name in names:
    doc=json.loads((ROOT/'notebooks'/name).read_text(encoding='utf-8'))
    for cell in doc['cells']:
        if cell['cell_type']=='code':cell['outputs']=[];cell['execution_count']=None
    (repro/'notebooks'/name).write_text(json.dumps(doc,ensure_ascii=False,indent=1),encoding='utf-8')
for name in ['build_h02_candidate.py','build_h02_reviewed.py']:
    shutil.copy2(ROOT/'scripts'/name,repro/'scripts'/name)
shutil.copy2(ROOT/'requirements.txt',repro/'requirements.txt')
shutil.copy2(ROOT/'docs/role01/protocol-reference.md',repro/'protocol-reference.md')
shutil.copy2(ROOT/'data/source-manifest.json',repro/'source-manifest.json')
artifact_list=[]
for path in sorted(OUT.rglob('*')):
    if path.is_file() and path.name not in ['data-manifest.json','CHECKSUMS.sha256']:
        artifact_list.append({'file':path.relative_to(OUT).as_posix(),'sha256':sha(path),'bytes':path.stat().st_size})
manifest={**previous,'data_version':VERSION,'status':'candidate_independently_verified_pending_H01_and_receipts',
          'generated_at_hong_kong':datetime.now(timezone(timedelta(hours=8))).isoformat(),
          'parent_manifest_sha256':sha(BASE/'data-manifest.json'),
          'change_note':'Numeric features and labels unchanged; new independent raw-data verification, processing and acceptance evidence.',
          'generation_command':command,'artifacts':artifact_list,
          'tests':{**previous['tests'],'independent_raw_recomputation_rows':len(features),
                   'independent_feature_and_label_columns':len(feature_names)+1,'independent_checks':'passed','v2_csv_roundtrip':'passed'},
          'team_receipts':'not_sent_no_receipts','formal_evaluation_approved':False,
          'next_status_target_hong_kong':'2026-11-08T21:00:00+08:00',
          'next_formal_revision_time':'pending_H01_and_team_agreement',
          'artifact_hash_note':'manifest and CHECKSUMS exclude self-referential hashes; core file hashes including manifest are in CHECKSUMS.sha256'}
manifest.pop('generated_at_utc',None)
(OUT/'data-manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
core=['snapshot_features.csv','snapshot_labels.csv','data-manifest.json','data-dictionary.md','coverage-and-cohorts.csv']
(OUT/'CHECKSUMS.sha256').write_text('\n'.join(sha(OUT/name)+'  '+name for name in core)+'\n',encoding='utf-8')
for a in artifact_list:assert sha(OUT/a['file'])==a['sha256']
bundle=ROOT/'outputs/public_learning/H02_candidate_v2.zip'
with ZipFile(bundle,'w',ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,'H02_candidate_v2/'+p.relative_to(OUT).as_posix())
with ZipFile(bundle) as z:assert z.testzip() is None
print('完成：',bundle)
print('状态：候选本地准备完成；待本人理解核验、H01确认与接收方回执。')

## 16｜你应该亲自回答的五题（没有记为已完成）

1. 为什么775笔商品匹配缺口，不能与160笔无批准订单、629笔金额资格不满足直接相加？
2. 本次选中多单客户的M是从哪几笔原始订单得来？运费加进去了吗？
3. 恰好在快照时刻批准的订单，属于哪边？把快照后移一秒会怎样？
4. 为什么4月21–23日没有批准记录，却不能断言那三天没有购买、也不能直接断言丢数？
5. “文件全部复算一致”是否表示H01已确认、最终测试已完成、组员已验收？

<details><summary>核对答案（建议先自己作答）</summary>

1. 前者是连接问题集合，后两者是顺序筛选中的互斥排除量；这些集合有交集，不能混加。
2. 将第6节实际选中客户的商品明细金额求和；不含运费，核对不同订单数。
3. 属于未来标签侧；后移一秒后进入历史侧。
4. 原始记录显示这641笔订单随后有批准时间。应调查事件时刻与记录过程，不能凭零日独断原因。
5. 否。可复现是技术验收；口径核定、源系统完整性、冻结和团队回执是不同事项。

</details>

你的下一步：对照第6节复算一名客户；读data-definition-review.md和HANDOFF.md；核验后在私有团队空间交接并收实际回执。没有新H01时先交候选及已核范围，不自行编造最终时间方案。